In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler,LabelEncoder
import pickle

In [ ]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
##drop irrelevent tables
#data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
#encode one categorical variable (means a variable which has only two values convert it to 0 and 1 like gender)
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
## encoding 'Geography'
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])# one hot encoder ko pura column chiye hota hai heading ke sath isliye double bracket vrna elements milenge sirf series mein
encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
#combined the encoded data to the main data
#data=data.drop(['Geography'],axis=1)
#data=pd.concat([data,encoded_df],axis=1)
data.head(25)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0
8,501,1,44,4,142051.07,2,0,1,74940.50,0,1.0,0.0,0.0
9,684,1,27,2,134603.88,1,1,1,71725.73,0,1.0,0.0,0.0


In [ ]:
#save the encoders and scaler 
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [ ]:
#divide the dataset into dependent and independent features
X=data.drop('Exited',axis=1)
y=data['Exited']
#split the data into training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42) #train test split x aur y ko split krdeta hai kuch data training ke liye aur kuch jispe model khudko test krega ki vo seekha ya nhi 0.2 ka mtlb hai 20% data testing ke liye  80 training ke liye
#scale these features
Scaler=StandardScaler()#ye Z value nikalne ke liye hota hai jaise statistics mein dekha tha values bht bdi hoti thi to usko z value mein convert krlete the z=x-u/sigma formula hota tha aur z table bhi hoti thi
X_train=Scaler.fit_transform(X_train) #isme learn krega nyi nyi values aur unko transform bhi krega
X_test=Scaler.transform(X_test) #isme sirf transform kyoki ye testing data hai


In [ ]:
#store this scaled data so you dont have to scale it again
with open('scaler.pickle','wb') as file:
    pickle.dump(Scaler,file)

ANN Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
X_train.shape[1]

12

In [ ]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), #hidden layer 1, input_shape=(X_train.shape[1],) isse input layer bnegi hr entry ke 12 features hain to 12 neurons 12, ka mtlb hai (12) 12 ki tuple single element tuple bnane ke liye , lgaya jata haiw
    Dense(32,activation='relu'), #hl2
    Dense(1,activation='sigmoid') #output layer
])

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:

opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [ ]:
#compile model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [ ]:
#setup tensorboard
log_dir="log/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
##setup earlystopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True) #ye loss agar kam nhi horha aur bdhne lgta hai to operation ko rok deta hai kyoki end tak jaane ka koi fayda nhi fir



In [ ]:
#train model
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/100
250/250 [==============================] - 3s 12ms/step - loss: 0.3408 - accuracy: 0.8612 - val_loss: 0.3393 - val_accuracy: 0.8605
Epoch 2/100
250/250 [==============================] - 3s 11ms/step - loss: 0.3354 - accuracy: 0.8590 - val_loss: 0.3514 - val_accuracy: 0.8550
Epoch 3/100
250/250 [==============================] - 2s 10ms/step - loss: 0.3374 - accuracy: 0.8585 - val_loss: 0.3438 - val_accuracy: 0.8580
Epoch 4/100
250/250 [==============================] - 2s 9ms/step - loss: 0.3333 - accuracy: 0.8637 - val_loss: 0.3407 - val_accuracy: 0.8555
Epoch 5/100
250/250 [==============================] - 2s 8ms/step - loss: 0.3337 - accuracy: 0.8639 - val_loss: 0.3392 - val_accuracy: 0.8585
Epoch 6/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3280 - accuracy: 0.8652 - val_loss: 0.3435 - val_accuracy: 0.8520
Epoch 7/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3272 - accuracy: 0.8640 - val_loss: 0.3476 - val_accuracy: 0.8

In [ ]:
model.save('model.h5')

e:\BankExitANNProject\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
#Load tensorboard extension
%load_ext tensorboard 

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:

%tensorboard --logdir log/fit

Reusing TensorBoard on port 6007 (pid 3208), started 0:18:04 ago. (Use '!kill 3208' to kill it.)

In [ ]:
#load pickle file
